# Lab 05 solution

In [ ]:
import json
from pathlib import Path
import requests
import pandas as pd

BASE = "https://api.worldbank.org/v2"
CACHE = Path("../../data/cache")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

COUNTRIES = ["AUS", "BRA", "CAN", "CHN", "DEU", "IND", "JPN",
             "KEN", "MEX", "NGA", "GBR", "USA", "VNM", "ZAF"]

In [ ]:
resp = requests.get(f"{BASE}/country/KEN/indicator/NE.EXP.GNFS.ZS",
                    params={"format": "json", "date": "2015:2023"}, timeout=30)
print(resp.status_code, resp.url)
meta, records = resp.json()
print(meta)
print(len(records), "records")

In [ ]:
print(json.dumps(records[0], indent=2))
# country name: record['country']['value']; iso3: record['countryiso3code']
# year: record['date'] (a string); value: record['value'] (may be None)
print(sum(r["value"] is None for r in records), "missing values")

In [ ]:
def fetch_indicator(countries, indicator, start, end):
    path = f"country/{';'.join(countries)}/indicator/{indicator}"
    params = {"format": "json", "date": f"{start}:{end}", "per_page": 1000}
    resp = requests.get(f"{BASE}/{path}", params=params, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    if len(payload) < 2 or not payload[1]:
        raise ValueError(f"API error: {payload}")
    if payload[0].get("pages", 1) > 1:
        raise ValueError("More than one page: increase per_page")
    return payload[1]

In [ ]:
# check
recs = fetch_indicator(["KEN", "NGA"], "NY.GDP.MKTP.CD", 2020, 2023)
assert len(recs) == 8
try:
    fetch_indicator(["XXX"], "NY.GDP.MKTP.CD", 2020, 2023)
    raise AssertionError("expected ValueError")
except ValueError:
    pass
print("Task 3 OK")

In [ ]:
def to_frame(records, name):
    return pd.DataFrame({
        "iso3": [r["countryiso3code"] for r in records],
        "country": [r["country"]["value"] for r in records],
        "year": [int(r["date"]) for r in records],
        name: [r["value"] for r in records],
    })

In [ ]:
INDICATORS = {
    "exports_pct_gdp": ("NE.EXP.GNFS.ZS", "wb_exports_pct_gdp.json"),
    "gdp_usd": ("NY.GDP.MKTP.CD", "wb_gdp_usd.json"),
}

frames = []
for name, (code, cache_file) in INDICATORS.items():
    try:
        recs = fetch_indicator(COUNTRIES, code, 2015, 2023)
        print(f"{name}: {len(recs)} records from API")
    except requests.RequestException as e:
        print(f"{name}: API unavailable ({e}); using cache")
        with open(CACHE / cache_file, encoding="utf-8") as f:
            recs = json.load(f)[1]
    frames.append(to_frame(recs, name))

wb = frames[0].merge(frames[1], on=["iso3", "country", "year"], how="outer")
wb = wb.sort_values(["iso3", "year"]).reset_index(drop=True)
wb.head()

In [ ]:
INDICATORS = {
    "exports_pct_gdp": ("NE.EXP.GNFS.ZS", "wb_exports_pct_gdp.json"),
    "gdp_usd": ("NY.GDP.MKTP.CD", "wb_gdp_usd.json"),
}
# TODO

In [ ]:
from datetime import date

missing = wb[wb[["exports_pct_gdp", "gdp_usd"]].isna().any(axis=1)]
print(len(missing), "rows with missing values")
print(missing[["country", "year"]])

wb["gdp_usd_bn"] = (wb["gdp_usd"] / 1e9).round(1)
wb["exports_usd_bn"] = (wb["exports_pct_gdp"] / 100 * wb["gdp_usd_bn"]).round(1)

wb.to_csv(OUT / "wb_indicators.csv", index=False)
metadata = {
    "source": BASE,
    "indicators": {k: v[0] for k, v in INDICATORS.items()},
    "retrieved": date.today().isoformat(),
    "rows": len(wb),
}
(OUT / "wb_indicators_metadata.json").write_text(json.dumps(metadata, indent=2))
print(metadata)

In [ ]:
latest = wb[wb["year"] == wb["year"].max()].sort_values("exports_pct_gdp", ascending=False)
latest[["country", "year", "exports_pct_gdp"]].head()